In [0]:
# ===========================================================================
# Notebook  : 02a_silver_events
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02a_silver_events
# Purpose   : bronze.{customer_id}.events_raw → hgi.silver.events
# NOTE: Reads from events_raw (BQ) → silver events
# Serverless : Works on 2X-Small (safe_spark_conf skips unsupported configs)
# Job Compute: Full perf configs applied automatically
# ===========================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/FINAL_pipeline_config.py

In [0]:
%run ./FINAL_silver_cdf_common.py

In [0]:
# CELL 2 ── Widgets
dbutils.widgets.text("customer_id",      "cust_001")
dbutils.widgets.text("starting_version", "0")
# starting_version='0' → replays full bronze history on first run
# After first run, checkpoint takes over (widget ignored)
# Set to a specific version number to skip already-processed history

customer_id      = dbutils.widgets.get("customer_id").strip().lower()
starting_version = dbutils.widgets.get("starting_version").strip()

In [0]:
# COMMAND ----------
# CELL 3 ── Object-specific constants
source_system = "bigquery"

# FIX (BUG 8): object_name must match the ACTUAL bronze table name suffix.
# bronze_table(customer_id, "events_raw") → bronze.cust_001.events_raw  ✅
# bronze_table(customer_id, "events")     → bronze.cust_001.crm_events  ❌ WRONG
object_name   = "events"

# Logical name used for column classification (matches STANDARD_COLS_BY_OBJECT key)
logical_name  = "events"

In [0]:
# COMMAND ----------
# CELL 4 ── Path and table resolution
bronze_full = bronze_table(customer_id, object_name)
# bronze_full = bronze.{customer_id}.events_raw  ✅

target_full = silver_table(logical_name)
# target_full = hgi.silver.events

chk_path = checkpoint_path("silver", "bigquery", customer_id, "events")

print(f"=== 02a Silver CDF: BIGQUERY Event ===")
print(f"  Customer         : {customer_id}")
print(f"  Bronze source    : {bronze_full}")
print(f"  Silver target    : {target_full}")
print(f"  Checkpoint       : {chk_path}")
print(f"  Starting version : {starting_version}")

In [0]:
# COMMAND ----------
# CELL 5 ── Create silver schema + table
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}.{silver_schema}")
create_silver_table(target_full, "events")

In [0]:

# COMMAND ----------
# CELL 6 ── Enable CDF on bronze source table
enable_cdf(bronze_full)

In [0]:

# COMMAND ----------
# CELL 7 ── Run CDF stream
# FIX: pass logical_name "events" (not "events_raw") so make_batch_processor
# looks up ("bigquery", "events") in STANDARD_COLS_BY_OBJECT correctly.
run_cdf_stream(bronze_full, target_full, "bigquery", logical_name,
               chk_path, starting_version)


In [0]:
# COMMAND ----------
# CELL 8 ── Post-load OPTIMIZE
spark.sql(f"OPTIMIZE {target_full}")
print(f"Silver CDF complete: {target_full}")
dbutils.notebook.exit("Success")
